## Data Cleaning

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 160)

# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
BASE_DIR = Path('..') / 'data' / 'bronze'

dataset_files = {
    'transactions': 'transactions_history_final.csv',
    'outlet_master': 'outlet_master.csv',
    'outlet_coordinates': 'outlet_coordinates.csv',
    'distributor_seasonality': 'distributor_seasonality_details.csv',
    'holidays': 'holiday_list.csv',
}

datasets = {}
dataset_headers = {}

for name, filename in dataset_files.items():
    file_path = BASE_DIR / filename
    datasets[name] = pd.read_csv(file_path)
    dataset_headers[name] = list(pd.read_csv(file_path, nrows=0).columns)

for name, headers in dataset_headers.items():
    print(f'{name}: {headers}')

transactions: ['Outlet_ID', 'Year', 'Month', 'Distributor_ID', 'SKU_ID', 'Volume_Liters', 'Total_Bill_Value']
outlet_master: ['Outlet_ID', 'Outlet_Size', 'Cooler_Count', 'Outlet_Type']
outlet_coordinates: ['Outlet_ID', 'Latitude', 'Longitude']
distributor_seasonality: ['Distributor_ID', 'Year', 'Month', 'Seasonality_Index']
holidays: ['Date', 'Holiday_Name', 'Holiday_Type']


In [6]:
datasets['transactions'].describe

<bound method NDFrame.describe of          Outlet_ID  Year  Month Distributor_ID  SKU_ID  Volume_Liters  Total_Bill_Value
0        OUT_19886  2024     12      DIST_S_02  SKU_10       5.897879       2177.632359
1        OUT_00837  2024      2      DIST_W_01  SKU_03      20.697364       7244.084814
2        OUT_15438  2025     12     DIST_NW_01  SKU_02      55.101801      13959.108787
3        OUT_12992  2025      1      DIST_C_01  SKU_07      24.063953      15641.548773
4        OUT_12334  2025      5      DIST_C_02  SKU_04      47.769665      15525.158656
...            ...   ...    ...            ...     ...            ...               ...
2376384  OUT_00825  2025      9      DIST_W_01  SKU_07      20.305661      13198.669945
2376385  OUT_15184  2024      8     DIST_NW_02  SKU_07      61.105053      39718.314060
2376386  OUT_09232  2025      3      DIST_C_01  SKU_09      14.505651      31912.430033
2376387  OUT_19669  2025     10      DIST_S_02  SKU_02      40.923617      10367.278716
2376388  OUT_01707  2024     10      DIST_W_01  SKU_07      10.426254       6777.044650

[2376389 rows x 7 columns]>

In [4]:
def _missing_mask(series: pd.Series) -> pd.Series:
    return series.isna() | series.astype(str).str.strip().eq('')


def duplicate_check(df: pd.DataFrame, primary_key):
    if primary_key is None or primary_key == "ALL":
        key_cols = list(df.columns)
    else:
        key_cols = [primary_key] if isinstance(primary_key, str) else list(primary_key)
    duplicate_mask = df.duplicated(subset=key_cols, keep=False)
    return df.loc[duplicate_mask].sort_values(key_cols).copy()


def null_check(df: pd.DataFrame, mandatory_fields):
    fields = [mandatory_fields] if isinstance(mandatory_fields, str) else list(mandatory_fields)
    mask = pd.Series(False, index=df.index)
    for field in fields:
        mask = mask | _missing_mask(df[field])
    return df.loc[mask, fields].copy()


def referential_integrity_check(df: pd.DataFrame, fk_cols, reference_df: pd.DataFrame, ref_cols=None):
    fk_cols = [fk_cols] if isinstance(fk_cols, str) else list(fk_cols)
    ref_cols = fk_cols if ref_cols is None else ([ref_cols] if isinstance(ref_cols, str) else list(ref_cols))

    df_keys = df[fk_cols].copy()
    ref_keys = reference_df[ref_cols].copy()

    for col in fk_cols:
        df_keys[col] = df_keys[col].astype(str).str.strip()
    for col in ref_cols:
        ref_keys[col] = ref_keys[col].astype(str).str.strip()

    merged = df_keys.merge(ref_keys.drop_duplicates(), left_on=fk_cols, right_on=ref_cols, how='left', indicator=True)
    invalid_mask = merged['_merge'].eq('left_only')
    return df.loc[invalid_mask].copy()


def value_range_check(df: pd.DataFrame, column: str, min_value=None, max_value=None):
    numeric = pd.to_numeric(df[column], errors='coerce')
    mask = pd.Series(False, index=df.index)
    if min_value is not None:
        mask = mask | numeric.lt(min_value)
    if max_value is not None:
        mask = mask | numeric.gt(max_value)
    mask = mask | (numeric.isna() & df[column].notna())
    return df.loc[mask, [column]].assign(parsed_numeric=numeric[mask])


def format_type_check(df: pd.DataFrame, column: str, *, expected_type=None, regex=None, date_format=None):
    series = df[column]
    mask = pd.Series(False, index=df.index)

    if expected_type == 'int':
        mask = mask | pd.to_numeric(series, errors='coerce').isna()
    elif expected_type == 'float':
        mask = mask | pd.to_numeric(series, errors='coerce').isna()
    elif expected_type == 'date':
        normalized = series.astype(str).str.strip().str[:10]
        normalized = normalized.where(series.notna(), series)
        parsed = pd.to_datetime(normalized, errors='coerce', format=date_format)
        mask = mask | parsed.isna()
    elif expected_type == 'string':
        mask = mask | series.isna()

    if regex is not None:
        mask = mask | ~series.astype(str).str.match(regex, na=False)

    return df.loc[mask, [column]].copy()


def summarize_result(title: str, result: pd.DataFrame):
    print(f'\n{title}: {len(result)} issue(s)')
    display(result.head(20))

## Configuring Checks

In [5]:
# Configure checks here
rules = {
    'transactions': {
        'primary_key': ['Outlet_ID', 'Year', 'Month', 'Distributor_ID', 'SKU_ID', 'Volume_Liters', 'Total_Bill_Value'],
        'mandatory_fields': ['Outlet_ID', 'Year', 'Month', 'Distributor_ID', 'SKU_ID', 'Volume_Liters', 'Total_Bill_Value'],
        'range_checks': {'Year': (2023, 2025), 'Month': (1, 12), 'Volume_Liters': (0, None), 'Total_Bill_Value': (0, None)},
        'format_checks': {'Outlet_ID': {'regex': r'^OUT_\d+$'}, 'Distributor_ID': {'regex': r'^DIST_[A-Z]+_\d+$'}, 'SKU_ID': {'regex': r'^SKU_\d+$'}},
    },
    'outlet_master': {
        'primary_key': 'Outlet_ID',
        'mandatory_fields': ['Outlet_ID','Cooler_Count', 'Outlet_Type'],
        'range_checks': {'Cooler_Count': (0, None)},
    },
    'outlet_coordinates': {
        'primary_key': 'Outlet_ID',
        'mandatory_fields': ['Outlet_ID', 'Latitude', 'Longitude'],
        'range_checks': {'Latitude': (-90, 90), 'Longitude': (-180, 180)},
    },
    'distributor_seasonality': {
        'primary_key': ['Distributor_ID', 'Year', 'Month'],
        'mandatory_fields': ['Distributor_ID', 'Year', 'Month', 'Seasonality_Index'],
        'range_checks': {'Year': (2023, 2025), 'Month': (1, 12)},
        'format_checks': {'Seasonality_Index': {'regex': r'^(Favorable|Moderate|Un-Favorable)$'}},
    },
    'holidays': {
        'primary_key': ['Date', 'Holiday_Name', 'Holiday_Type'],
        'mandatory_fields': ['Date'],
        'format_checks': {'Date': {'expected_type': 'date', 'date_format': '%Y-%m-%d'}},
    },
}

results = {}

for dataset_name, config in rules.items():
    df = datasets[dataset_name]
    print(f'\n=== {dataset_name.upper()} ===')

    dupes = duplicate_check(df, config['primary_key'])
    results[(dataset_name, 'duplicates')] = dupes
    summarize_result('Duplicate records', dupes)

    nulls = null_check(df, config['mandatory_fields'])
    results[(dataset_name, 'nulls')] = nulls
    summarize_result('Null / empty mandatory fields', nulls)

    # Small, non-blocking check: for holidays only, flag empty optional fields (Holiday_Name, Holiday_Type)
    if dataset_name == 'holidays':
        for opt_col in ('Holiday_Name', 'Holiday_Type'):
            if opt_col in df.columns:
                mask = df[opt_col].astype(str).str.strip().eq('') | df[opt_col].isna()
                issues = df.loc[mask, [opt_col]].copy()
            else:
                issues = df.iloc[0:0].copy()
            results[(dataset_name, f'empty_{opt_col}')] = issues
            summarize_result(f'Empty {opt_col}', issues)

    return_mask = pd.Series(False, index=df.index)
    # if dataset_name == 'transactions':
    #     return_mask = (df['Volume_Liters'] < 0) & (df['Total_Bill_Value'] < 0)

    for column, bounds in config.get('range_checks', {}).items():
        low, high = bounds
        issues = value_range_check(df, column, low, high)
        if dataset_name == 'transactions' and column in ('Volume_Liters', 'Total_Bill_Value'):
            issues = issues.loc[~return_mask.reindex(issues.index, fill_value=False)]
        results[(dataset_name, f'range_{column}')] = issues
        summarize_result(f'Range check for {column}', issues)

    for column, fmt in config.get('format_checks', {}).items():
        issues = format_type_check(df, column, **fmt)
        results[(dataset_name, f'format_{column}')] = issues
        summarize_result(f'Format / type check for {column}', issues)

# Referential integrity checks
ri_checks = [
    ('transactions', ['Outlet_ID'], 'outlet_master', ['Outlet_ID']),
    ('transactions', ['Outlet_ID'], 'outlet_coordinates', ['Outlet_ID']),
    ('transactions', ['Distributor_ID', 'Year', 'Month'], 'distributor_seasonality', ['Distributor_ID', 'Year', 'Month']),
]

for child_name, fk_cols, parent_name, ref_cols in ri_checks:
    issues = referential_integrity_check(datasets[child_name], fk_cols, datasets[parent_name], ref_cols)
    results[(child_name, f'ref_{parent_name}')] = issues
    summarize_result(f'Referential integrity: {child_name} -> {parent_name}', issues)


=== TRANSACTIONS ===

Duplicate records: 0 issue(s)


,Outlet_ID,Year,Month,Distributor_ID,SKU_ID,Volume_Liters,Total_Bill_Value



Null / empty mandatory fields: 0 issue(s)


,Outlet_ID,Year,Month,Distributor_ID,SKU_ID,Volume_Liters,Total_Bill_Value



Range check for Year: 0 issue(s)


...

,Outlet_ID,Year,Month,Distributor_ID,SKU_ID,Volume_Liters,Total_Bill_Value



Referential integrity: transactions -> outlet_coordinates: 0 issue(s)


,Outlet_ID,Year,Month,Distributor_ID,SKU_ID,Volume_Liters,Total_Bill_Value



Referential integrity: transactions -> distributor_seasonality: 0 issue(s)


,Outlet_ID,Year,Month,Distributor_ID,SKU_ID,Volume_Liters,Total_Bill_Value


## Cleaning

In [ ]:
from collections import defaultdict

SILVER_DIR = Path('..') / 'data' / 'silver'
REJECTED_DIR = Path('..') / 'data' / 'rejected'
SILVER_DIR.mkdir(parents=True, exist_ok=True)
REJECTED_DIR.mkdir(parents=True, exist_ok=True)

ri_checks = globals().get(
    'ri_checks',
    [
        ('transactions', ['Outlet_ID'], 'outlet_master', ['Outlet_ID']),
        ('transactions', ['Outlet_ID'], 'outlet_coordinates', ['Outlet_ID']),
        ('transactions', ['Distributor_ID', 'Year', 'Month'], 'distributor_seasonality', ['Distributor_ID', 'Year', 'Month']),
    ],
)

def _add_reason(reason_map, indices, reason):
    for idx in indices:
        reason_map[idx].append(reason)

def build_rejections(df: pd.DataFrame, config: dict, dataset_name: str):

    reasons = defaultdict(list)
    return_mask = pd.Series(False, index=df.index)
    if dataset_name == 'transactions':
        return_mask = (df['Volume_Liters'] < 0) & (df['Total_Bill_Value'] < 0)

    dupes = duplicate_check(df, config['primary_key'])
    _add_reason(reasons, dupes.index, 'duplicate_key')

    nulls = null_check(df, config['mandatory_fields'])
    _add_reason(reasons, nulls.index, 'missing_mandatory')

    for column, bounds in config.get('range_checks', {}).items():
        low, high = bounds
        issues = value_range_check(df, column, low, high)
        if dataset_name == 'transactions' and column in ('Volume_Liters', 'Total_Bill_Value'):
            issues = issues.loc[~return_mask.reindex(issues.index, fill_value=False)]
        _add_reason(reasons, issues.index, f'out_of_range:{column}')

    for column, fmt in config.get('format_checks', {}).items():
        issues = format_type_check(df, column, **fmt)
        _add_reason(reasons, issues.index, f'format:{column}')

    for child_name, fk_cols, parent_name, ref_cols in ri_checks:
        if child_name != dataset_name:
            continue
        issues = referential_integrity_check(df, fk_cols, datasets[parent_name], ref_cols)
        _add_reason(reasons, issues.index, f'missing_fk:{parent_name}')

    reject_indices = sorted(reasons.keys())
    if reject_indices:
        rejected_df = df.loc[reject_indices].copy()
        rejected_df['rejection_reasons'] = ["; ".join(reasons[idx]) for idx in reject_indices]
    else:
        rejected_df = df.iloc[0:0].copy()
        rejected_df['rejection_reasons'] = pd.Series(dtype=str)

    cleaned_df = df.drop(index=reject_indices).copy()
    return cleaned_df, rejected_df

silver_outputs = {}
rejected_outputs = {}

for dataset_name, config in rules.items():
    df = datasets[dataset_name]

    # Apply specific cleaning for 'outlet_master' dataset
    if dataset_name == 'outlet_master':
        if 'Outlet_Type' in df.columns:
            df['Outlet_Type'] = df['Outlet_Type'].astype(str).str.strip()
            df['Outlet_Type'] = df['Outlet_Type'].replace({'Grocry': 'Grocery', 'Bakry': 'Bakery'})

        if 'Outlet_Size' in df.columns:
            df['Outlet_Size'] = df['Outlet_Size'].replace({'Small': 'small'})

        # Impute missing Cooler_Count values
        if 'Cooler_Count' in df.columns:
            df['Cooler_Count'] = df.groupby(['Outlet_Type', 'Outlet_Size'])['Cooler_Count'].transform(lambda x: x.fillna(x.median()))

    cleaned_df, rejected_df = build_rejections(df, config, dataset_name)
    silver_outputs[dataset_name] = cleaned_df
    rejected_outputs[dataset_name] = rejected_df

    cleaned_path = SILVER_DIR / f"{dataset_name}.csv"
    rejected_path = REJECTED_DIR / f"{dataset_name}_rejected.csv"
    cleaned_df.to_csv(cleaned_path, index=False)
    rejected_df.to_csv(rejected_path, index=False)

    print(f"{dataset_name}: {len(cleaned_df)} cleaned, {len(rejected_df)} rejected")

transactions: 2376389 cleaned, 0 rejected
outlet_master: 19804 cleaned, 196 rejected
outlet_coordinates: 20000 cleaned, 0 rejected
distributor_seasonality: 360 cleaned, 0 rejected
holidays: 163 cleaned, 186 rejected


### Aggregate transactions data to monthly frequency

In [7]:
transactions_monthly_agg = silver_outputs['transactions'].groupby(['Outlet_ID', 'Year', 'Month']).agg(
    monthly_volume=('Volume_Liters', 'sum'),
    monthly_bill_value=('Total_Bill_Value', 'sum')
).reset_index()

# Save the aggregated data to a new CSV file in the SILVER_DIR
aggregated_transactions_path = SILVER_DIR / "transactions_monthly_aggregated.csv"
transactions_monthly_agg.to_csv(aggregated_transactions_path, index=False)

print(f"Aggregated transactions data saved to: {aggregated_transactions_path}")
display(transactions_monthly_agg.head())

Aggregated transactions data saved to: /content/drive/MyDrive/data_storm/data/silver/transactions_monthly_aggregated.csv


,Outlet_ID,Year,Month,monthly_volume,monthly_bill_value
0,OUT_00001,2023,3,1390.918670,398271.118522
1,OUT_00001,2023,6,382.687950,100870.517231
2,OUT_00001,2023,8,552.713934,147318.575355
3,OUT_00001,2023,10,541.861465,107725.488528
4,OUT_00001,2023,11,223.594015,53456.527694


### Engineer a monthly holiday_count predictive feature

In [8]:
holidays_df = silver_outputs['holidays'].copy()
# Ensure 'Date' column is in datetime format
holidays_df['Date'] = pd.to_datetime(holidays_df['Date'])

# Extract Year and Month
holidays_df['Year'] = holidays_df['Date'].dt.year
holidays_df['Month'] = holidays_df['Date'].dt.month

# Group by Year and Month to count holidays
monthly_holiday_count = holidays_df.groupby(['Year', 'Month']).size().reset_index(name='holiday_count')

# Save the aggregated data to a new CSV file in the SILVER_DIR
holiday_count_path = SILVER_DIR / "monthly_holiday_count.csv"
monthly_holiday_count.to_csv(holiday_count_path, index=False)

print(f"Monthly holiday count data saved to: {holiday_count_path}")
display(monthly_holiday_count.head())

Monthly holiday count data saved to: /content/drive/MyDrive/data_storm/data/silver/monthly_holiday_count.csv


,Year,Month,holiday_count
0,2023,2,6
1,2023,5,6
2,2024,1,7
3,2024,2,9
4,2024,3,5
